In [7]:
import pandas as pd
import xarray as xr
import numpy as np

In [8]:
def load_region_matrix(csv_path, lat_size=360, lon_size=720):
    df = pd.read_csv(csv_path)

    region_matrix = np.empty((lat_size, lon_size), dtype=object)
    region_matrix[:] = None  # 默认填 None

    for _, r in df.iterrows():
        I = int(r["I"]) - 1
        J = int(r["J"]) - 1
        region_matrix[I, J] = r["Rall"]

    return region_matrix


In [9]:
def split_variable_streaming(da, region_matrix, regions, output_path):
    """
    使用逐时间片写出（streaming）的方式拆分变量，避免一次性加载巨大数组。
    """

    time = da["time"]
    lat  = da["lat"]
    lon  = da["lon"]

    # 输出 Dataset 初始化（空）
    ds_out = xr.Dataset()

    # 先为每个 region 创建变量（只定义 shape，不写数据）
    for reg in regions:
        var_name = f"{reg}_{da.name}"
        ds_out[var_name] = xr.DataArray(
            np.full((len(time), len(lat), len(lon)), np.nan, dtype=np.float32),
            coords={"time": time, "lat": lat, "lon": lon},
            dims=("time", "lat", "lon")
        )

    # ---- 逐时间片处理 ----
    for ti, t in enumerate(time.values):
        print(f"Processing time = {t}")

        # 只读这个时间片 → 是 360×720，不会爆内存
        da_t = da.sel(time=t).values  # shape = (lat, lon)

        for reg in regions:
            var_name = f"{reg}_{da.name}"

            # mask
            masked = np.where(region_matrix == reg, da_t, np.nan)

            # 写入 Dataset 的对应时间层
            ds_out[var_name][ti, :, :] = masked

    # ---- 最终写入文件 ----
    ds_out.to_netcdf(output_path)
    print(f"[Saved] {output_path}")

In [10]:
def write_region_split_nc(output_path, region_dict):
    ds = xr.Dataset()
    for reg, da in region_dict.items():
        varname = da.name.replace(" ", "_")
        ds[varname] = da
    ds.to_netcdf(output_path)
    print(f"[Saved] {output_path}")

In [11]:
def run_full_split(csv_path, nc_path, output_prefix):
    # 载入 CSV→region_matrix
    region_matrix = load_region_matrix(csv_path, lat_size=360, lon_size=720)   
    ds = xr.open_dataset(nc_path)

    # 收集地域名称
    regions = set()
    for r in region_matrix.flatten():
        if isinstance(r, str) and r.strip() != "":
            regions.add(r)
    regions = sorted(regions)

    variable_groups = [
        ("region_agri", "region", "agri"),
        ("region_forest", "region", "forest"),
        ("region_grassland", "region", "grassland"),
        ("basin_agri", "basin", "agri"),
        ("basin_forest", "basin", "forest"),
        ("basin_grassland", "basin", "grassland"),
    ]

    for varname, mode, landtype in variable_groups:
        print(f"\n========== Splitting {varname} ==========")

        if varname not in ds:
            print(f"{varname} missing, skip.")
            continue

        da = ds[varname]
        da.name = f"{mode}_{landtype}"

        output_file = f"{output_prefix}_{mode}_{landtype}_17regions.nc"

        # 使用 streaming 函数
        split_variable_streaming(da, region_matrix, regions, output_file)


In [12]:
CSV_PATH = "../../CSV/RIJ.csv"
NC_PATH  = "../../NC/compare.nc"
OUTPUT_PREFIX = "split"

run_full_split(CSV_PATH, NC_PATH, OUTPUT_PREFIX)


========== Splitting region_agri ==========


MemoryError: Unable to allocate 10.9 MiB for an array with shape (11, 360, 720) and data type float32